# レッスン9: ファイル読み書きとモジュール 💾📦

プログラムの計算結果は、実行が終わると消えます。**残したいものはファイルに書く** — これが今日の前半。後半は、コードが大きくなったときに**ファイルを分割して整理する仕組み(モジュール)**。どちらも実務コードの土台です。

---
## 9-1. ファイルに書く — with open

Pythonのファイル操作は `with open(...) as f:` が定型文です。

In [ ]:
with open("result.txt", "w") as f:       # "w" = write(書き込み)。ファイルが無ければ作られる
    f.write("実験結果\n")                   # \n = 改行
    f.write("温度: 25.3\n")
    f.write("湿度: 61\n")

print("保存しました")   # Colabでは左のフォルダ📁に result.txt が現れます

`with` を使うと、ブロックを抜けるときにファイルが自動で閉じられます(閉じ忘れ事故の防止)。実務ではほぼ100% `with` を使います。

## 9-2. ファイルを読む

In [ ]:
with open("result.txt", "r") as f:       # "r" = read(読み込み)
    naiyou = f.read()                     # 全文を1つの文字列として読む
print(naiyou)

with open("result.txt", "r") as f:
    for line in f:                        # 1行ずつのループも定番
        print("行:", line.strip())        # strip()で行末の改行を除去(セット技)

## 9-3. CSVファイル — 表データの共通言語

カンマ区切りのテキスト(CSV)はExcelでもPythonでも開ける、実務データ交換の共通形式です。レッスン7の `split` / `join` がそのまま活きます。

In [ ]:
# 書き出し: 計算結果の表をCSVに
kion = [22.5, 24.1, 26.8, 25.0]
with open("kion.csv", "w") as f:
    f.write("hour,temp\n")                  # 1行目は見出し(ヘッダー)
    for i, t in enumerate(kion):             # enumerate = 番号と値を同時に取るループ技
        f.write(f"{i},{t}\n")

# 読み込み: CSVを辞書のリストに復元
rows = []
with open("kion.csv", "r") as f:
    header = f.readline()                    # 1行目(見出し)を読み飛ばす
    for line in f:
        parts = line.strip().split(",")
        rows.append({"hour": int(parts[0]), "temp": float(parts[1])})

print(rows)

## 9-4. JSON — 辞書をそのまま保存する

辞書やリストの入れ子構造を保存するなら **JSON** 形式。Web API・設定ファイル・アプリ間のデータ受け渡しと、実務での遭遇率はトップクラスです。標準ライブラリ `json` で辞書⇔ファイルを直接変換できます。

In [ ]:
import json

materials = {
    "concrete": {"lam": 1.6, "rho": 2200, "c": 880},
    "glass":    {"lam": 1.0, "rho": 2500, "c": 750},
}

with open("materials.json", "w") as f:
    json.dump(materials, f, indent=2)        # 辞書 → ファイル

with open("materials.json", "r") as f:
    loaded = json.load(f)                    # ファイル → 辞書に完全復元

print(loaded["concrete"]["lam"])

---
## 9-5. モジュール — コードをファイルに分けて再利用する

`import` の正体は「**別のPythonファイルを読み込む**」です。numpyも実は大量の.pyファイルの集まり。つまり自分のコードも同じように部品化できます。

Colab上でファイル分割を体験してみましょう(`%%writefile` はセルの内容をファイルに保存するColabの命令)。

In [ ]:
%%writefile netsu.py
# netsu.py — 熱計算の自作モジュール

def kakusanritsu(lam, rho, c):
    """熱拡散率 a = λ/(ρc) [m²/s]"""
    return lam / (rho * c)

def dt_max(dx, a):
    """陽解法の安定条件によるdt上限 [s]"""
    if dx <= 0 or a <= 0:
        raise ValueError("dxとaは正の値が必要です")
    return dx ** 2 / (2 * a)

In [ ]:
import netsu                    # ← 自分で作ったファイルをimport!

a = netsu.kakusanritsu(1.6, 2200, 880)
print("a =", a)
print("dt上限 =", netsu.dt_max(0.01, a), "秒")

これが実務のコードベースの姿です。**大きなプログラム = 小さなモジュールの集合体**。「calculation.py に計算、plot.py に描画、main.py に全体の流れ」のように役割ごとにファイルを分けます。

### 標準ライブラリ ちょい見せツアー

Pythonには最初から便利なモジュールが大量に入っています(標準ライブラリ)。

In [ ]:
import math
print(math.sqrt(2), math.pi)          # 平方根、円周率

import random
print(random.randint(1, 6))           # 1〜6のランダム整数(サイコロ)

from datetime import date             # from モジュール import 名前 という書き方
print(date.today())                   # 今日の日付

---
## 💼 実務メモ: import文は「コードの依存関係の一覧表」

実務でコードを読むとき、プロはまず**ファイル冒頭のimport文**を見ます。「このコードは何に依存して何をするのか」の目次だからです。書く側の作法として、importは(1)標準ライブラリ (2)外部ライブラリ (3)自作モジュール の順にファイル冒頭へまとめます。

---
## ✏️ 練習問題 9-A

九九の表(1×1〜9×9)を `kuku.txt` に書き出してください。1行に「2 x 3 = 6」のような形式で81行。書き出したら読み込んで最初の数行を表示して確認。

ヒント: forループの中にforループ(二重ループ)を書きます。

In [ ]:
# ここにコードを書いてください


---
## ✏️ 練習問題 9-B 【実務風: 設定ファイル】

シミュレーションの設定辞書(`L`, `n`, `dt`, `material` の4キー。値は自由)を作り、`config.json` に保存 → 読み込んで、`f"壁厚{L}m を {n}分割、dt={dt}s"` のように表示してください。

「設定をコードから分離してJSONに置く」のは実務ソフトの標準設計です。

In [ ]:
# ここにコードを書いてください


---
## ✏️ 練習問題 9-C 【モジュール化】

レッスン3-Bの `c_to_f`(摂氏→華氏)と、逆変換 `f_to_c` の2つの関数を持つモジュール `ondo.py` を `%%writefile` で作り、importして両方使ってください。

- 確認: `f_to_c(c_to_f(25))` が 25 に戻れば変換ペアは正しい(これは**往復テスト**という実務のテスト技法です)

In [ ]:
# ここにコードを書いてください (セル1: %%writefile ondo.py)


In [ ]:
# ここにコードを書いてください (セル2: importして使う)


---
## 🎉 レッスン9はここまで!

**今日覚えたこと:**
1. `with open(path, "w"/"r") as f:` がファイル操作の定型文
2. CSVは `split`/`join` で、辞書の保存は `json.dump`/`json.load`
3. `import` の正体は「他の.pyファイルの読み込み」— 自作モジュールでコードを分割
4. `enumerate`、二重ループ、標準ライブラリ(math/random/datetime)

**次回 → レッスン10: クラスとオブジェクト指向**(実務コードの主要な書き方)

---
### 💡 答え

<details>
<summary>クリックで表示</summary>

```python
# 9-A
with open("kuku.txt", "w") as f:
    for i in range(1, 10):
        for j in range(1, 10):
            f.write(f"{i} x {j} = {i*j}\n")

with open("kuku.txt", "r") as f:
    for k, line in enumerate(f):
        if k < 5:
            print(line.strip())

# 9-B
import json
config = {"L": 0.3, "n": 30, "dt": 20, "material": "concrete"}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
with open("config.json", "r") as f:
    cfg = json.load(f)
print(f"壁厚{cfg['L']}m を {cfg['n']}分割、dt={cfg['dt']}s")

# 9-C (セル1)
%%writefile ondo.py
def c_to_f(t_c):
    return 9 / 5 * t_c + 32

def f_to_c(t_f):
    return (t_f - 32) * 5 / 9

# 9-C (セル2)
import ondo
print(ondo.c_to_f(25))            # 77.0
print(ondo.f_to_c(ondo.c_to_f(25)))   # 25.0 に戻れば正しい
```
</details>